# Run on Kaggle

Trains and evaluates runs of the matrix on a Kaggle GPU. Every step only calls code from the repo, at the same commit as the Colab runs (`COMMIT` below), so the training and evaluation code is identical.

Settings (right-hand panel): **Accelerator: GPU T4 x2** (the code uses one GPU) and **Internet: On** (needs a phone-verified account). Then **Save Version > Save & Run All (Commit)**: it runs in the background, with no browser needed, for up to 12 hours.

Results land in `/kaggle/working/export/` (`raw/`, `run_folders/`, `logs/`, `runs.csv`), the same layout as the Drive `results/` folder. Download the version's output and import it with `python -m scripts.collect_results --from <folder>`.

Each version starts from nothing: there is no resume across versions, so give it one whole run.

In [ ]:
RUN_ARGS = "--models resnet18 --strategies augmix --seeds 2"  # scripts/run_all.sh filters
REPO_URL = "https://github.com/aysha0559/Image-Classification-with-ResNet-18.git"
COMMIT = "7d334eb220c19c79e32d50b073bbe992028ef27f"  # the commit every Colab run used

In [ ]:
import os

REPO_DIR = "/tmp/kaggle-run/cifar10-robustness-study"
DATA_DIR = "/tmp/kaggle-run/data"
WORK = "/kaggle/working"
EXPORT = f"{WORK}/export"
LOG = f"{EXPORT}/logs/kaggle.console.log"
TEE = f"2>&1 | tee -a {LOG}"

os.makedirs(f"{EXPORT}/logs", exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.environ["DATA_DIR"] = DATA_DIR
os.environ["RUNS_DIR"] = f"{WORK}/runs"
# Outside the repo: rebuilding the tracked results/runs.csv inside it would mark runs git_dirty.
os.environ["RESULTS_DIR"] = f"{WORK}/results"
os.environ["PYTHONUNBUFFERED"] = "1"


def require_ok(step):
    """Stop the run if the last `!` command failed."""
    assert get_ipython().user_ns["_exit_code"] == 0, f"{step} failed; read the output above"

Stops here if there is no GPU: `device: auto` would otherwise train on the CPU without a word.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch; print(torch.__version__, torch.cuda.get_device_name(0))"
require_ok("the GPU check (Settings > Accelerator > GPU T4 x2)")

In [ ]:
!git clone {REPO_URL} {REPO_DIR}
require_ok("cloning the repo (Settings > Internet must be On)")
%cd {REPO_DIR}
!git checkout --detach {COMMIT}
require_ok("checking out the commit")
!pip install -q -r requirements.txt
require_ok("pip install")
!bash -c "echo === kaggle session $(date -u +%FT%TZ) commit $(git rev-parse HEAD) {TEE}"

Downloads CIFAR-10 and CIFAR-10-C (about 2.9 GB) and verifies every file against `splits/cifar10c_checksums.json`.

In [ ]:
!python -c "from src.data.cifar10 import get_test_dataset; get_test_dataset('{DATA_DIR}')"
require_ok("getting CIFAR-10")
!bash -o pipefail -c "python -m scripts.download_cifar10c --data-dir {DATA_DIR} {TEE}"
require_ok("getting CIFAR-10-C")
!rm -f {DATA_DIR}/CIFAR-10-C.tar

In [ ]:
dirty = !git status --porcelain --untracked-files=no
assert not dirty, f"tracked files changed, so runs would be marked git_dirty: {dirty}"
env = f"RESULTS_SYNC_DIR={EXPORT}"
!bash -o pipefail -c "{env} bash scripts/run_all.sh {RUN_ARGS} {TEE}"
require_ok("the runs")
!find {EXPORT} -type f | sort